<a href="https://colab.research.google.com/github/SattamAltwaim/StarX/blob/main/experiments/12_sketch_to_3d_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sketch to 3D - try the fine-tuned model

Give it a drawing, get a mesh you can spin around and download.

This runs the TripoSR checkpoint fine-tuned in notebook 11 - stock architecture, all 419 M parameters trained on 105k synthetic sketches of real CAD parts. On held-out designs it reaches **29.5 dB PSNR** and **0.90 silhouette IoU**, against 15.5 dB and 0.39 for the pretrained model it started from.

Three ways in, set by `INPUT` in the configuration cell:

- **`"test"`** - a held-out design the model never trained on. Start here; it is the honest case.
- **`"upload"`** - your own image. A clean line drawing works directly; a photo or a render gets edge-detected first, the same way the training data was made.
- **`"draw"`** - draw one in the browser.

Then: four rendered views, an interactive mesh you can orbit, a downloadable OBJ, a turntable GIF, and a side-by-side against stock TripoSR.

Needs a GPU runtime (Runtime -> Change runtime type -> T4 is plenty) and the checkpoint on your Drive at `StarX/runs/sketch_ft_ddp/`.

In [ ]:
# Setup: clone the repo and the pinned TripoSR, install this notebook's
# dependencies, mount Drive (the checkpoint lives there).
import os
import subprocess
import sys

BRANCH = "main"
TRIPOSR_COMMIT = "107cefdc244c39106fa830359024f6a2f1c78871"
NOTEBOOK_ID = "12"

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    REPO_DIR = "/content/StarX"
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--branch", BRANCH,
             "https://github.com/SattamAltwaim/StarX.git", REPO_DIR],
            check=True,
        )
    TRIPOSR_DIR = "/content/TripoSR"
else:
    def _find_repo():
        for start in (globals().get("__vsc_ipynb_file__"), os.getcwd()):
            if not start:
                continue
            path = os.path.abspath(
                os.path.dirname(start) if os.path.isfile(start) else start
            )
            while path != os.path.dirname(path):
                if os.path.exists(os.path.join(path, "starx", "pins.py")):
                    return path
                path = os.path.dirname(path)
        home = os.path.join(os.path.expanduser("~"), "StarX")
        if os.path.exists(os.path.join(home, "starx", "pins.py")):
            return home
        raise RuntimeError("could not locate the StarX repo")

    REPO_DIR = _find_repo()
    TRIPOSR_DIR = os.path.join(REPO_DIR, "third_party", "TripoSR")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if not os.path.exists(TRIPOSR_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git",
         TRIPOSR_DIR], check=True,
    )
subprocess.run(["git", "-C", TRIPOSR_DIR, "checkout", "-q", TRIPOSR_COMMIT], check=True)

from starx import pins

assert pins.TRIPOSR_COMMIT == TRIPOSR_COMMIT, "notebook pin out of sync"
if pins.PIP_PINS[NOTEBOOK_ID]:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pins.PIP_PINS[NOTEBOOK_ID]],
        check=True,
    )
if IN_COLAB and pins.PIP_UNINSTALL.get(NOTEBOOK_ID):
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-q", "-y",
         *pins.PIP_UNINSTALL[NOTEBOOK_ID]],
        check=False, capture_output=True,
    )

from starx import colab as scolab

DRIVE = scolab.mount_drive()
report = scolab.setup_report()

In [ ]:
# Configuration - every knob for this notebook lives here.
import io
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import Image, display
from PIL import Image as PILImage
from tqdm.auto import tqdm

from starx import cameras, shards, synth, viz
from starx import eval as seval
from starx import model as smodel
from starx import train as strain
from starx.config import CAMERA_DISTANCE, FOVY_DEG, StarXConfig, shard_dir

# where to get the sketch: "test" | "upload" | "draw"
INPUT = "test"
EDGE_DETECT_UPLOAD = True   # an uploaded photo/render -> Sobel edges first;
                            # set False if you upload a clean line drawing
TEST_SHARD, TEST_INDEX, TEST_VIEW = 0, 3, 0   # which held-out design

MC_RESOLUTION = 256         # marching-cubes grid; 128 is faster, 320 finer
MC_THRESHOLD = 25.0         # density isosurface level

DRIVE_RUN = (DRIVE / "StarX" / "runs" / "sketch_ft_ddp") if DRIVE else Path(
    os.path.join(REPO_DIR, "data", "StarX", "runs", "sketch_ft_ddp")
)
CHECKPOINT = DRIVE_RUN / "model_only_step12000.pt"
OUT_DIR = Path("/content/starx_out") if IN_COLAB else Path(REPO_DIR) / "demo_out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = StarXConfig(
    drive_root=str(DRIVE / "StarX") if DRIVE else os.path.join(REPO_DIR, "data", "StarX"),
    sketch_size=512, edge_blur_sigma=1.2, edge_gain=3.0, edge_bg=1.0,
    gt_size=256, eval_chunk=131072,
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP, _ = strain.pick_amp(DEVICE)

DRAW_WIDGET = """
<div style="font-family:system-ui">
<canvas id="pad" width="512" height="512"
        style="border:1px solid #bbb;border-radius:6px;background:#fff;cursor:crosshair"></canvas>
<div style="margin-top:6px">
  <button onclick="clearPad()">clear</button>
  <button onclick="savePad()">Save</button>
  <span id="pad-status" style="margin-left:8px;color:#666"></span>
</div></div>
<script>
const c=document.getElementById('pad'),x=c.getContext('2d');
x.fillStyle='#fff';x.fillRect(0,0,512,512);x.strokeStyle='#000';x.lineWidth=3;
x.lineCap='round';x.lineJoin='round';let drawing=false;
c.onmousedown=e=>{drawing=true;x.beginPath();x.moveTo(e.offsetX,e.offsetY);};
c.onmousemove=e=>{if(drawing){x.lineTo(e.offsetX,e.offsetY);x.stroke();}};
c.onmouseup=()=>drawing=false;c.onmouseleave=()=>drawing=false;
function clearPad(){x.fillStyle='#fff';x.fillRect(0,0,512,512);
  document.getElementById('pad-status').innerText='';}
function savePad(){google.colab.kernel.invokeFunction('notebook.save_drawing',
  [c.toDataURL('image/png')],{});
  document.getElementById('pad-status').innerText='saved - run the next cell';}
</script>"""

print(f"device {DEVICE}, autocast {AMP}")
print(f"checkpoint: {CHECKPOINT}")
print(f"  {'FOUND' if CHECKPOINT.exists() else 'MISSING - will fall back to stock TripoSR'}")
print(f"input mode: {INPUT}")

In [ ]:
# Load the fine-tuned model. The checkpoint on Drive is inference-only -
# weights, no optimizer state - so it is 1.6 GB rather than 4.7 GB.
model = smodel.load_pretrained_tsr(TRIPOSR_DIR, device="cpu")
assert model.image_tokenizer.model.embeddings.patch_embeddings.projection.in_channels == 3

if CHECKPOINT.exists():
    state = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
    missing, unexpected = model.load_state_dict(state["model"], strict=False)
    print(f"loaded step {state['step']} from {CHECKPOINT.name}")
    print(f"  {len(state['model'])} tensors, "
          f"{sum(v.numel() for v in state['model'].values()):,} parameters")
    if missing or unexpected:
        print(f"  missing {len(missing)}, unexpected {len(unexpected)} "
              f"(non-persistent buffers are expected here)")
else:
    print(f"NO CHECKPOINT at {CHECKPOINT}\n"
          f"  running the STOCK pretrained model instead - it has never seen a\n"
          f"  line drawing, so expect poor results. Point CHECKPOINT at the file\n"
          f"  under {DRIVE_RUN} on your Drive.")

model = model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad_(False)
print(f"\nready on {DEVICE}, autocast {AMP}")

In [ ]:
# Get a sketch. Three ways - set INPUT above to choose.
#
# The model wants a 512px image of DARK LINES ON A LIGHT PAGE. Anything
# else (a photo, a shaded render, a dark-background drawing) goes through
# the same Sobel edge detector the training data was built with, so it
# arrives looking like what the model was taught on.
import base64

from IPython.display import HTML


def to_sketch(image, edge_detect: bool) -> torch.Tensor:
    """PIL image -> (3, 512, 512) float in [0, 1], dark lines on light."""
    rgb = np.asarray(image.convert("RGB"))
    if edge_detect:
        return synth.sobel_sketch(rgb, cfg.sketch_size, cfg.edge_blur_sigma,
                                  cfg.edge_gain, cfg.edge_bg)
    square = PILImage.fromarray(rgb).resize(
        (cfg.sketch_size, cfg.sketch_size), PILImage.LANCZOS
    )
    gray = torch.from_numpy(np.asarray(square).astype(np.float32) / 255.0).mean(dim=-1)
    if gray.mean() < 0.5:            # light lines on a dark page: invert
        gray = 1.0 - gray
    return gray[None].repeat(3, 1, 1)


raw, sketch, SOURCE_NAME = None, None, "input"

if INPUT == "upload":
    from google.colab import files as colab_files

    print("choose a sketch (png/jpg) - a photo works too, it gets edge-detected")
    uploaded = colab_files.upload()
    name = next(iter(uploaded))
    raw = PILImage.open(io.BytesIO(uploaded[name]))
    SOURCE_NAME = Path(name).stem
    sketch = to_sketch(raw, EDGE_DETECT_UPLOAD)

elif INPUT == "draw":
    # The canvas posts its PNG back to the kernel through a registered
    # callback. Draw, press Save, then RE-RUN this cell: the saved file is
    # picked up on the second pass.
    from google.colab import output as colab_output

    drawing_path = OUT_DIR / "drawing.png"

    def _save_drawing(data_url):
        drawing_path.write_bytes(base64.b64decode(data_url.split(",", 1)[1]))

    colab_output.register_callback("notebook.save_drawing", _save_drawing)

    if drawing_path.exists():
        raw = PILImage.open(drawing_path)
        SOURCE_NAME = "drawing"
        sketch = to_sketch(raw, edge_detect=False)
        print(f"using {drawing_path} - delete it (or press Save again) to redraw")
    else:
        display(HTML(DRAW_WIDGET))
        print("draw above, press Save, then RE-RUN this cell")

else:  # "test" - a held-out design the model has never trained on
    tar = shards.list_done_shards(shard_dir(cfg, "test"))[TEST_SHARD]
    for n, sample in enumerate(shards.iter_shard(tar)):
        if n == TEST_INDEX:
            break
    SOURCE_NAME = sample["design_id"]
    raw = PILImage.fromarray(sample["views"][TEST_VIEW])
    sketch = synth.sobel_sketch(sample["views"][TEST_VIEW], cfg.sketch_size,
                                cfg.edge_blur_sigma, cfg.edge_gain, cfg.edge_bg)
    print(f"held-out design {SOURCE_NAME}, view {TEST_VIEW}")

if sketch is not None:
    panels = [(raw, "source")] if raw is not None else []
    panels.append((sketch[0], f"model input {tuple(sketch.shape)}"))
    fig, axes = plt.subplots(1, len(panels), figsize=(3.6 * len(panels), 3.6))
    for ax, (img, title) in zip(np.atleast_1d(axes), panels):
        ax.imshow(img, cmap="gray" if getattr(img, "ndim", 3) == 2 else None,
                  vmin=0 if getattr(img, "ndim", 3) == 2 else None,
                  vmax=1 if getattr(img, "ndim", 3) == 2 else None)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.show()

In [ ]:
# Reconstruct. One forward pass through the encoder gives a triplane scene
# code - that IS the 3D object; everything below is just looking at it.
model.renderer.set_chunk_size(cfg.eval_chunk)
with torch.no_grad():
    with torch.autocast("cuda", dtype=AMP):
        scene_code = smodel.encode_sketches(model, sketch[None].to(DEVICE))[0]
    scene_code = scene_code.float()
print(f"scene code {tuple(scene_code.shape)} in {DEVICE}")

# four views of it, including ones the sketch never showed
angles = [0.0, 60.0, 140.0, 250.0]
fig, axes = plt.subplots(1, 1 + len(angles), figsize=(3.0 * (1 + len(angles)), 3.2))
axes[0].imshow(sketch[0], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("your sketch", fontsize=10)
with torch.no_grad():
    for ax, azimuth in zip(axes[1:], angles):
        c2w = cameras.build_spherical_c2w(azimuth, 20.0, CAMERA_DISTANCE)
        rays_o, rays_d = cameras.rays_full(c2w, FOVY_DEG, 256)
        rgb, alpha = strain.render_rays(
            model, scene_code, rays_o.to(DEVICE), rays_d.to(DEVICE)
        )
        ax.imshow(strain.composite_over_gray(rgb, alpha, 1.0).clamp(0, 1).cpu().numpy())
        ax.set_title(f"azimuth {azimuth:.0f}", fontsize=10)
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
# The mesh: marching cubes over the predicted density field, then an
# interactive plotly view - drag to orbit, scroll to zoom. Finally an OBJ
# you can download and open anywhere.
mesh = seval.extract_mesh(model, scene_code, res=MC_RESOLUTION, threshold=MC_THRESHOLD)
if mesh is None:
    raise RuntimeError(
        "nothing crossed the density threshold - the model saw no object here. "
        "Try a cleaner sketch, or lower MC_THRESHOLD."
    )
print(f"{len(mesh.vertices):,} vertices, {len(mesh.faces):,} faces, "
      f"watertight={mesh.is_watertight}")

v, f = np.asarray(mesh.vertices), np.asarray(mesh.faces)
figure = go.Figure(
    data=[go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        color="#4C9BE8", opacity=1.0, flatshading=True,
        lighting=dict(ambient=0.45, diffuse=0.9, specular=0.25, roughness=0.6),
        lightposition=dict(x=2, y=2, z=3),
    )]
)
figure.update_layout(
    title=f"{SOURCE_NAME} - drag to orbit, scroll to zoom",
    scene=dict(aspectmode="data", xaxis_visible=False,
               yaxis_visible=False, zaxis_visible=False),
    margin=dict(l=0, r=0, t=32, b=0), height=560,
)
figure.show()

obj_path = OUT_DIR / f"{SOURCE_NAME}.obj"
mesh.export(obj_path)
print(f"\nwrote {obj_path}  ({obj_path.stat().st_size / 1024:.0f} KB)")
if IN_COLAB:
    from google.colab import files as colab_files

    colab_files.download(str(obj_path))

In [ ]:
# A turntable, because a still image cannot show whether the thing is
# actually solid. 36 renders around the object, straight from the NeRF.
frames = []
model.renderer.set_chunk_size(cfg.eval_chunk)
with torch.no_grad():
    for azimuth in tqdm(range(0, 360, 10), desc="turntable"):
        c2w = cameras.build_spherical_c2w(float(azimuth), 20.0, CAMERA_DISTANCE)
        rays_o, rays_d = cameras.rays_full(c2w, FOVY_DEG, 256)
        rgb, alpha = strain.render_rays(
            model, scene_code, rays_o.to(DEVICE), rays_d.to(DEVICE)
        )
        frame = strain.composite_over_gray(rgb, alpha, 1.0).clamp(0, 1)
        frames.append((frame.cpu().numpy() * 255).astype(np.uint8))

gif_path = OUT_DIR / "turntable.gif"
viz.turntable_gif(frames, gif_path, fps=12)
print(f"wrote {gif_path}")
display(Image(filename=str(gif_path)))

In [ ]:
# Stock TripoSR vs the fine-tuned model, same sketch, same cameras.
# The pretrained checkpoint has never seen a line drawing; this is the
# gap the fine-tuning closed.
stock = smodel.load_pretrained_tsr(TRIPOSR_DIR, device=DEVICE)
stock.renderer.set_chunk_size(cfg.eval_chunk)
with torch.no_grad():
    with torch.autocast("cuda", dtype=AMP):
        stock_code = smodel.encode_sketches(stock, sketch[None].to(DEVICE))[0].float()

fig, axes = plt.subplots(2, 4, figsize=(13, 6.6))
for row, (net, code, name) in enumerate(
    ((stock, stock_code, "stock TripoSR"), (model, scene_code, "fine-tuned"))
):
    axes[row, 0].imshow(sketch[0], cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_ylabel(name, fontsize=11)
    for col, azimuth in enumerate([0.0, 90.0, 200.0]):
        c2w = cameras.build_spherical_c2w(azimuth, 20.0, CAMERA_DISTANCE)
        rays_o, rays_d = cameras.rays_full(c2w, FOVY_DEG, 256)
        with torch.no_grad():
            rgb, alpha = strain.render_rays(
                net, code, rays_o.to(DEVICE), rays_d.to(DEVICE)
            )
        axes[row, 1 + col].imshow(
            strain.composite_over_gray(rgb, alpha, 0.5).clamp(0, 1).cpu().numpy()
        )
        if row == 0:
            axes[row, 1 + col].set_title(f"azimuth {azimuth:.0f}", fontsize=9)
axes[0, 0].set_title("input sketch", fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()

del stock, stock_code
torch.cuda.empty_cache()

## What to try next

- **Draw badly on purpose.** Wobbly lines, a missing edge, a wrong perspective. The model was trained on clean Sobel edges of rendered CAD parts, so this is where its limits show.
- **Feed it a photograph.** Set the preprocessing cell to edge-detect and watch what a real object becomes.
- **Turn the object further.** The turntable is the honest test - anything that looks right from the input view and wrong at 90 degrees is a billboard, not a reconstruction.
- **Compare against stock.** The comparison cell runs the same sketch through the pretrained checkpoint. On held-out designs the fine-tuned model scores +14 dB PSNR and 0.90 vs 0.39 silhouette IoU, and the difference is usually obvious by eye.

The mesh you downloaded is a normalized unit-ish object centred at the origin - open it in Blender, MeshLab, or drop it straight back into `project11`'s viewer.